# Day 6: Advanced Analytics & Risk Metrics
## Bluestock Mutual Fund Analytics Capstone

This notebook performs comprehensive risk and performance analytics, benchmark comparisons, and investor demographics analysis.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from scipy.stats import linregress

# Set style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# File paths
ROOT = Path('..').resolve()
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW = ROOT / 'data' / 'raw'
REPORTS = ROOT / 'reports'

REPORTS.mkdir(parents=True, exist_ok=True)

print(f'Root: {ROOT}')
print(f'Data folder: {DATA_PROCESSED if DATA_PROCESSED.exists() else DATA_RAW}')

In [ ]:
# Data loading function
def load_dataset(filename):
    """Load dataset from processed or raw folder."""
    paths = [DATA_PROCESSED / filename, DATA_RAW / filename]
    for path in paths:
        if path.exists():
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
            return df
    print(f'Warning: {filename} not found')
    return pd.DataFrame()

# Load datasets
print('Loading datasets...')
fund_master = load_dataset('01_fund_master.csv')
nav_history = load_dataset('02_nav_history.csv')
scheme_performance = load_dataset('07_scheme_performance.csv')
benchmark_indices = load_dataset('10_benchmark_indices.csv')
investor_transactions = load_dataset('08_investor_transactions.csv')

print(f'Fund Master: {len(fund_master)} records')
print(f'NAV History: {len(nav_history)} records')
print(f'Scheme Performance: {len(scheme_performance)} records')
print(f'Benchmark Indices: {len(benchmark_indices)} records')
print(f'Investor Transactions: {len(investor_transactions)} records')

## Risk & Performance Analytics

### Calculate Key Risk Metrics

In [ ]:
# Constants
RISK_FREE_RATE = 0.065  # 6.5% annual risk-free rate
TRADING_DAYS = 252

def prepare_nav_returns(nav_df):
    """Prepare daily returns from NAV history."""
    if nav_df.empty:
        return pd.DataFrame()
    
    nav = nav_df.copy()
    
    # Ensure date column is datetime
    if 'date' in nav.columns:
        nav['date'] = pd.to_datetime(nav['date'], errors='coerce')
    elif 'nav_date' in nav.columns:
        nav['date'] = pd.to_datetime(nav['nav_date'], errors='coerce')
    
    # Find NAV value column
    nav_col = next((col for col in nav.columns if 'nav' in col.lower() and col != 'date'), None)
    if nav_col:
        nav['nav'] = pd.to_numeric(nav[nav_col], errors='coerce')
    
    # Identify scheme/fund column
    scheme_col = next((col for col in nav.columns if 'amfi' in col.lower() or 'scheme' in col.lower() or 'code' in col.lower()), None)
    if scheme_col:
        nav['amfi_code'] = nav[scheme_col]
    
    nav = nav[['amfi_code', 'date', 'nav']].dropna()
    nav = nav.sort_values(['amfi_code', 'date'])
    nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
    
    return nav

nav_returns = prepare_nav_returns(nav_history)
print(f'NAV Returns prepared: {len(nav_returns)} records')
if not nav_returns.empty:
    print(nav_returns.head())

In [ ]:
def calculate_sharpe_ratio(returns):
    """Calculate annualized Sharpe ratio."""
    returns = returns.dropna()
    if len(returns) < 2 or returns.std() == 0:
        return np.nan
    excess_return = returns.mean() * TRADING_DAYS - RISK_FREE_RATE
    volatility = returns.std() * np.sqrt(TRADING_DAYS)
    return excess_return / volatility if volatility != 0 else np.nan

def calculate_sortino_ratio(returns):
    """Calculate annualized Sortino ratio."""
    returns = returns.dropna()
    if len(returns) < 2:
        return np.nan
    excess_return = returns.mean() * TRADING_DAYS - RISK_FREE_RATE
    downside = returns[returns < 0]
    if len(downside) == 0:
        return np.nan
    downside_dev = np.sqrt((downside ** 2).mean()) * np.sqrt(TRADING_DAYS)
    return excess_return / downside_dev if downside_dev != 0 else np.nan

def calculate_var(returns, confidence=0.95):
    """Calculate Value at Risk at given confidence level."""
    returns = returns.dropna()
    if len(returns) < 2:
        return np.nan
    return -np.percentile(returns, (1 - confidence) * 100)

def calculate_max_drawdown(nav_series, dates):
    """Calculate maximum drawdown and period."""
    if len(nav_series) < 2:
        return np.nan, None, None
    
    cumulative_max = nav_series.cummax()
    drawdown = (nav_series - cumulative_max) / cumulative_max
    max_dd = drawdown.min()
    
    max_dd_idx = drawdown.idxmin()
    peak_idx = nav_series[:max_dd_idx].idxmax()
    
    peak_date = dates.iloc[peak_idx] if peak_idx < len(dates) else None
    trough_date = dates.iloc[max_dd_idx] if max_dd_idx < len(dates) else None
    
    return -max_dd, peak_date, trough_date

print('Risk metric functions defined')

In [ ]:
# Calculate metrics for each fund
def calculate_fund_metrics(nav_returns_df):
    """Calculate all risk metrics for all funds."""
    if nav_returns_df.empty:
        return pd.DataFrame()
    
    metrics = []
    
    for amfi_code, group in nav_returns_df.groupby('amfi_code'):
        returns = group['daily_return'].dropna()
        nav_series = group.set_index('date')['nav'].sort_index()
        dates = group.set_index('date').index
        
        if len(returns) < 20:  # Need sufficient data
            continue
        
        max_dd, peak_date, trough_date = calculate_max_drawdown(nav_series, dates)
        
        metrics.append({
            'amfi_code': amfi_code,
            'annualized_return': returns.mean() * TRADING_DAYS,
            'annualized_volatility': returns.std() * np.sqrt(TRADING_DAYS),
            'sharpe_ratio': calculate_sharpe_ratio(returns),
            'sortino_ratio': calculate_sortino_ratio(returns),
            'var_95': calculate_var(returns, 0.95),
            'max_drawdown': max_dd,
            'drawdown_start_date': peak_date,
            'drawdown_end_date': trough_date,
            'num_observations': len(returns)
        })
    
    return pd.DataFrame(metrics)

print('Calculating fund metrics...')
fund_metrics = calculate_fund_metrics(nav_returns)
print(f'Metrics calculated for {len(fund_metrics)} funds')
print(fund_metrics.head())

In [ ]:
# Create ranking tables
def create_ranking_tables(fund_metrics_df, fund_master_df, scheme_performance_df):
    """Create ranking tables for all funds."""
    if fund_metrics_df.empty:
        return pd.DataFrame()
    
    rankings = fund_metrics_df.copy()
    
    # Merge with scheme information
    if not scheme_performance_df.empty:
        scheme_info = scheme_performance_df[['amfi_code', 'scheme_name', 'fund_house', 'category']].drop_duplicates()
        rankings = rankings.merge(scheme_info, on='amfi_code', how='left')
    elif not fund_master_df.empty:
        scheme_info = fund_master_df[['amfi_code', 'scheme_name', 'fund_house']].drop_duplicates()
        rankings = rankings.merge(scheme_info, on='amfi_code', how='left')
    
    # Calculate percentile ranks
    rankings['sharpe_rank'] = rankings['sharpe_ratio'].rank(pct=True, ascending=False) * 100
    rankings['sortino_rank'] = rankings['sortino_ratio'].rank(pct=True, ascending=False) * 100
    rankings['var_rank'] = rankings['var_95'].rank(pct=True, ascending=True) * 100
    rankings['volatility_rank'] = rankings['annualized_volatility'].rank(pct=True, ascending=True) * 100
    
    return rankings

fund_sharpe_ranks = create_ranking_tables(fund_metrics, fund_master, scheme_performance)
print(f'Ranking tables created: {len(fund_sharpe_ranks)} funds')
print(fund_sharpe_ranks[['amfi_code', 'scheme_name', 'sharpe_ratio', 'sortino_ratio', 'var_95', 'sharpe_rank']].head(10))

In [ ]:
# Calculate alpha and beta
def calculate_alpha_beta(nav_returns_df, benchmark_df):
    """Calculate alpha and beta for each fund vs each benchmark."""
    if nav_returns_df.empty or benchmark_df.empty:
        return pd.DataFrame()
    
    bench = benchmark_df.copy()
    
    # Prepare benchmark data
    if 'date' in bench.columns:
        bench['date'] = pd.to_datetime(bench['date'], errors='coerce')
    
    # Find index name and close value columns
    index_col = next((col for col in bench.columns if 'index' in col.lower() or 'benchmark' in col.lower()), None)
    close_col = next((col for col in bench.columns if 'close' in col.lower() or 'value' in col.lower()), None)
    
    if index_col and close_col:
        bench['index_name'] = bench[index_col]
        bench['close'] = pd.to_numeric(bench[close_col], errors='coerce')
    
    bench = bench[['date', 'index_name', 'close']].dropna()
    bench = bench.sort_values(['index_name', 'date'])
    bench['bench_return'] = bench.groupby('index_name')['close'].pct_change()
    
    alpha_beta_data = []
    
    for index_name in bench['index_name'].unique():
        bench_data = bench[bench['index_name'] == index_name][['date', 'bench_return']].dropna()
        
        for amfi_code, group in nav_returns_df.groupby('amfi_code'):
            fund_data = group[['date', 'daily_return']].dropna()
            
            # Merge on date
            merged = fund_data.merge(bench_data, on='date', how='inner')
            
            if len(merged) < 20:
                continue
            
            x = merged['bench_return'].values
            y = merged['daily_return'].values
            
            if np.std(x) == 0:
                continue
            
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            
            tracking_error = np.std(y - (intercept + slope * x)) * np.sqrt(TRADING_DAYS)
            
            alpha_beta_data.append({
                'amfi_code': amfi_code,
                'benchmark': index_name,
                'alpha': intercept * TRADING_DAYS,
                'beta': slope,
                'tracking_error': tracking_error,
                'r_squared': r_value ** 2
            })
    
    return pd.DataFrame(alpha_beta_data)

print('Calculating alpha and beta...')
alpha_beta = calculate_alpha_beta(nav_returns, benchmark_indices)
print(f'Alpha/Beta calculated: {len(alpha_beta)} records')
if not alpha_beta.empty:
    print(alpha_beta.head())

## Benchmark Analysis

### Compare fund returns with major benchmarks

In [ ]:
# Prepare benchmark returns
def prepare_benchmark_returns(benchmark_df):
    """Prepare benchmark returns for comparison."""
    if benchmark_df.empty:
        return pd.DataFrame()
    
    bench = benchmark_df.copy()
    
    if 'date' in bench.columns:
        bench['date'] = pd.to_datetime(bench['date'], errors='coerce')
    
    index_col = next((col for col in bench.columns if 'index' in col.lower() or 'benchmark' in col.lower()), None)
    close_col = next((col for col in bench.columns if 'close' in col.lower() or 'value' in col.lower()), None)
    
    if index_col and close_col:
        bench['index_name'] = bench[index_col]
        bench['close'] = pd.to_numeric(bench[close_col], errors='coerce')
    
    bench = bench[['date', 'index_name', 'close']].dropna().sort_values(['index_name', 'date'])
    bench['return'] = bench.groupby('index_name')['close'].pct_change()
    
    return bench

benchmark_returns = prepare_benchmark_returns(benchmark_indices)
print(f'Benchmark returns prepared: {len(benchmark_returns)} records')

In [ ]:
# Calculate rolling correlation
def calculate_rolling_correlation(nav_returns_df, benchmark_df, window=60):
    """Calculate rolling correlation between funds and benchmarks."""
    if nav_returns_df.empty or benchmark_df.empty:
        return pd.DataFrame()
    
    correlations = []
    
    for index_name in benchmark_df['index_name'].unique():
        bench_data = benchmark_df[benchmark_df['index_name'] == index_name][['date', 'return']].dropna()
        
        for amfi_code, group in nav_returns_df.groupby('amfi_code'):
            fund_data = group[['date', 'daily_return']].dropna()
            
            merged = fund_data.merge(bench_data, on='date', how='inner')
            
            if len(merged) < window:
                continue
            
            merged = merged.set_index('date').sort_index()
            rolling_corr = merged['daily_return'].rolling(window).corr(merged['return'])
            
            avg_corr = rolling_corr.mean()
            
            correlations.append({
                'amfi_code': amfi_code,
                'benchmark': index_name,
                'avg_correlation': avg_corr,
                'correlation_std': rolling_corr.std()
            })
    
    return pd.DataFrame(correlations)

print('Calculating rolling correlations...')
rolling_correlations = calculate_rolling_correlation(nav_returns, benchmark_returns)
print(f'Rolling correlations calculated: {len(rolling_correlations)} records')

## Investor Demographics Analysis

### Analyze investor profiles and behavior patterns

In [ ]:
# Prepare investor transaction data
def prepare_investor_data(transactions_df):
    """Prepare and clean investor transaction data."""
    if transactions_df.empty:
        return pd.DataFrame()
    
    inv = transactions_df.copy()
    
    # Standardize column names
    inv.columns = inv.columns.str.lower().str.replace(' ', '_')
    
    return inv

investor_data = prepare_investor_data(investor_transactions)
print(f'Investor data prepared: {len(investor_data)} records')
if not investor_data.empty:
    print('Columns:', investor_data.columns.tolist())
    print(investor_data.head())

In [ ]:
# Age distribution analysis
age_analysis = None
income_analysis = None
state_analysis = None
sip_lumpsum_analysis = None
city_tier_analysis = None
gender_analysis = None

if not investor_data.empty:
    # Age distribution
    age_cols = [col for col in investor_data.columns if 'age' in col.lower()]
    if age_cols:
        age_col = age_cols[0]
        investor_data[age_col] = pd.to_numeric(investor_data[age_col], errors='coerce')
        age_analysis = investor_data.groupby(pd.cut(investor_data[age_col], bins=[0, 25, 35, 45, 55, 150])).size()
        print('Age Distribution:')
        print(age_analysis)
    
    # Income distribution
    income_cols = [col for col in investor_data.columns if 'income' in col.lower()]
    if income_cols:
        income_col = income_cols[0]
        if investor_data[income_col].dtype == 'object':
            income_analysis = investor_data[income_col].value_counts()
        else:
            income_analysis = investor_data.groupby(pd.cut(investor_data[income_col], bins=5)).size()
        print('\nIncome Distribution:')
        print(income_analysis)
    
    # State-wise analysis
    state_cols = [col for col in investor_data.columns if 'state' in col.lower()]
    if state_cols:
        state_col = state_cols[0]
        state_analysis = investor_data[state_col].value_counts().head(15)
        print('\nTop 15 States:')
        print(state_analysis)
    
    # SIP vs Lumpsum
    sip_cols = [col for col in investor_data.columns if 'sip' in col.lower() or 'transaction_type' in col.lower()]
    if sip_cols:
        sip_col = sip_cols[0]
        sip_lumpsum_analysis = investor_data[sip_col].value_counts()
        print('\nSIP vs Lumpsum:')
        print(sip_lumpsum_analysis)
    
    # City tier analysis
    city_cols = [col for col in investor_data.columns if 'city' in col.lower() or 'tier' in col.lower()]
    if city_cols:
        city_col = city_cols[0]
        city_tier_analysis = investor_data[city_col].value_counts()
        print('\nCity Tier Distribution:')
        print(city_tier_analysis)
    
    # Gender analysis
    gender_cols = [col for col in investor_data.columns if 'gender' in col.lower()]
    if gender_cols:
        gender_col = gender_cols[0]
        gender_analysis = investor_data[gender_col].value_counts()
        print('\nGender Distribution:')
        print(gender_analysis)

## Visualizations

### Professional charts using Plotly and Seaborn

In [ ]:
# 1. Risk vs Return Scatter Plot
if not fund_sharpe_ranks.empty:
    fig_scatter = go.Figure()
    
    fig_scatter.add_trace(go.Scatter(
        x=fund_sharpe_ranks['annualized_volatility'] * 100,
        y=fund_sharpe_ranks['annualized_return'] * 100,
        mode='markers',
        marker=dict(
            size=8,
            color=fund_sharpe_ranks['sharpe_ratio'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='Sharpe Ratio'),
            line=dict(width=0.5, color='white')
        ),
        text=[f"{row['scheme_name']}<br>Return: {row['annualized_return']*100:.2f}%<br>Volatility: {row['annualized_volatility']*100:.2f}%<br>Sharpe: {row['sharpe_ratio']:.2f}" 
              for _, row in fund_sharpe_ranks.iterrows()],
        hovertemplate='<b>%{text}</b><extra></extra>',
        name='Funds'
    ))
    
    fig_scatter.update_layout(
        title='Risk vs Return Analysis - All Funds',
        xaxis_title='Volatility (Annualized %)',
        yaxis_title='Return (Annualized %)',
        template='plotly_white',
        height=600,
        width=1000,
        hovermode='closest'
    )
    
    fig_scatter.write_html(str(REPORTS / 'risk_return_scatter.html'))
    print('Risk vs Return scatter plot saved')

In [ ]:
# 2. Sharpe Ratio Ranking Chart (Top 20)
if not fund_sharpe_ranks.empty:
    top_sharpe = fund_sharpe_ranks.nlargest(20, 'sharpe_ratio')[['scheme_name', 'sharpe_ratio', 'fund_house']].copy()
    
    fig_sharpe = go.Figure(data=[
        go.Bar(
            y=top_sharpe['scheme_name'],
            x=top_sharpe['sharpe_ratio'],
            orientation='h',
            marker=dict(
                color=top_sharpe['sharpe_ratio'],
                colorscale='RdYlGn',
                showscale=False
            ),
            text=[f"{val:.2f}" for val in top_sharpe['sharpe_ratio']],
            textposition='outside',
            hovertemplate='<b>%{y}</b><br>Sharpe: %{x:.2f}<extra></extra>'
        )
    ])
    
    fig_sharpe.update_layout(
        title='Top 20 Funds by Sharpe Ratio',
        xaxis_title='Sharpe Ratio',
        template='plotly_white',
        height=700,
        width=1000,
        showlegend=False
    )
    
    fig_sharpe.write_html(str(REPORTS / 'sharpe_ratio_ranking.html'))
    print('Sharpe ratio ranking chart saved')

In [ ]:
# 3. Alpha vs Beta Scatter
if not alpha_beta.empty:
    # Get Nifty 50 comparison
    nifty_50_data = alpha_beta[alpha_beta['benchmark'].str.contains('50|Nifty 50', case=False, na=False)]
    
    if not nifty_50_data.empty:
        fig_ab = go.Figure()
        
        fig_ab.add_trace(go.Scatter(
            x=nifty_50_data['beta'],
            y=nifty_50_data['alpha'] * 100,
            mode='markers',
            marker=dict(
                size=8,
                color=nifty_50_data['r_squared'],
                colorscale='Plasma',
                showscale=True,
                colorbar=dict(title='R²'),
                line=dict(width=0.5, color='white')
            ),
            text=nifty_50_data['amfi_code'],
            hovertemplate='<b>%{text}</b><br>Beta: %{x:.2f}<br>Alpha: %{y:.2f}%<extra></extra>'
        ))
        
        fig_ab.update_layout(
            title='Alpha vs Beta Analysis (vs Nifty 50)',
            xaxis_title='Beta (Systematic Risk)',
            yaxis_title='Alpha (Excess Return %)',
            template='plotly_white',
            height=600,
            width=1000
        )
        
        fig_ab.write_html(str(REPORTS / 'alpha_beta_comparison.html'))
        print('Alpha vs Beta chart saved')

In [ ]:
# 4. State-wise Investor Distribution
if state_analysis is not None and len(state_analysis) > 0:
    fig_state = go.Figure(data=[
        go.Bar(
            x=state_analysis.index,
            y=state_analysis.values,
            marker=dict(
                color=state_analysis.values,
                colorscale='Blues',
                showscale=False
            ),
            hovertemplate='<b>%{x}</b><br>Investors: %{y}<extra></extra>'
        )
    ])
    
    fig_state.update_layout(
        title='Top 15 States - Investor Distribution',
        xaxis_title='State',
        yaxis_title='Number of Investors',
        template='plotly_white',
        height=500,
        width=1000,
        xaxis_tickangle=-45
    )
    
    fig_state.write_html(str(REPORTS / 'state_distribution.html'))
    print('State distribution chart saved')

In [ ]:
# 5. SIP vs Lumpsum Pie Chart
if sip_lumpsum_analysis is not None and len(sip_lumpsum_analysis) > 0:
    fig_sip = go.Figure(data=[go.Pie(
        labels=sip_lumpsum_analysis.index,
        values=sip_lumpsum_analysis.values,
        hole=0.3,
        hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
    )])
    
    fig_sip.update_layout(
        title='SIP vs Lumpsum Investment Distribution',
        height=600,
        width=800
    )
    
    fig_sip.write_html(str(REPORTS / 'sip_lumpsum_distribution.html'))
    print('SIP vs Lumpsum chart saved')

In [ ]:
# 6. Income Distribution
if income_analysis is not None and len(income_analysis) > 0:
    fig_income = go.Figure(data=[
        go.Bar(
            x=income_analysis.index.astype(str),
            y=income_analysis.values,
            marker=dict(
                color=income_analysis.values,
                colorscale='Greens',
                showscale=False
            ),
            hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
        )
    ])
    
    fig_income.update_layout(
        title='Income Bracket Distribution',
        xaxis_title='Income Bracket',
        yaxis_title='Number of Investors',
        template='plotly_white',
        height=500,
        width=1000
    )
    
    fig_income.write_html(str(REPORTS / 'income_distribution.html'))
    print('Income distribution chart saved')

In [ ]:
# 7. Benchmark Performance Comparison
if not benchmark_returns.empty:
    bench_summary = benchmark_returns.groupby('index_name')['return'].agg(['mean', 'std']).reset_index()
    
    if not bench_summary.empty:
        fig_bench = make_subplots(
            rows=1, cols=2,
            subplot_titles=('Average Daily Returns', 'Volatility'),
            specs=[[{'type':'bar'}, {'type':'bar'}]]
        )
        
        fig_bench.add_trace(
            go.Bar(
                x=bench_summary['index_name'],
                y=bench_summary['mean'] * 252 * 100,
                name='Annualized Return',
                marker_color='lightblue',
                hovertemplate='<b>%{x}</b><br>Return: %{y:.2f}%<extra></extra>'
            ),
            row=1, col=1
        )
        
        fig_bench.add_trace(
            go.Bar(
                x=bench_summary['index_name'],
                y=bench_summary['std'] * np.sqrt(252) * 100,
                name='Annualized Volatility',
                marker_color='lightcoral',
                hovertemplate='<b>%{x}</b><br>Volatility: %{y:.2f}%<extra></extra>'
            ),
            row=1, col=2
        )
        
        fig_bench.update_xaxes(title_text='Benchmark', row=1, col=1)
        fig_bench.update_xaxes(title_text='Benchmark', row=1, col=2)
        fig_bench.update_yaxes(title_text='Return (%)', row=1, col=1)
        fig_bench.update_yaxes(title_text='Volatility (%)', row=1, col=2)
        
        fig_bench.update_layout(
            title='Benchmark Performance Comparison',
            height=500,
            width=1200,
            template='plotly_white',
            showlegend=False
        )
        
        fig_bench.write_html(str(REPORTS / 'benchmark_comparison.html'))
        print('Benchmark comparison chart saved')

## Business Insights

### Key findings and recommendations

In [ ]:
insights = """
# Day 6 Advanced Analytics - Business Insights

## Risk & Performance Summary

### Key Findings
"""

if not fund_sharpe_ranks.empty:
    top_sharpe_fund = fund_sharpe_ranks.nlargest(1, 'sharpe_ratio').iloc[0]
    avg_sharpe = fund_sharpe_ranks['sharpe_ratio'].mean()
    
    insights += f"""

1. **Best Risk-Adjusted Returns**: {top_sharpe_fund['scheme_name']} ({top_sharpe_fund['fund_house']})
   - Sharpe Ratio: {top_sharpe_fund['sharpe_ratio']:.2f}
   - Annual Return: {top_sharpe_fund['annualized_return']*100:.2f}%
   - Volatility: {top_sharpe_fund['annualized_volatility']*100:.2f}%
   - Average Fund Sharpe Ratio: {avg_sharpe:.2f}

2. **Downside Risk Analysis**:
   - Average Maximum Drawdown: {fund_sharpe_ranks['max_drawdown'].mean()*100:.2f}%
   - Most Resilient Fund: {fund_sharpe_ranks.nsmallest(1, 'max_drawdown').iloc[0]['scheme_name']}
   - Worst Drawdown: {fund_sharpe_ranks['max_drawdown'].max()*100:.2f}%

3. **Value at Risk (VaR 95%)**:
   - Average Daily VaR: {fund_sharpe_ranks['var_95'].mean()*100:.2f}%
   - This means 95% of trading days have losses less than this amount

4. **Volatility Profile**:
   - Average Annual Volatility: {fund_sharpe_ranks['annualized_volatility'].mean()*100:.2f}%
   - Lowest Risk Fund: {fund_sharpe_ranks.nsmallest(1, 'annualized_volatility').iloc[0]['scheme_name']}
   - Highest Risk Fund: {fund_sharpe_ranks.nlargest(1, 'annualized_volatility').iloc[0]['scheme_name']}
"""

if not alpha_beta.empty:
    nifty_50_ab = alpha_beta[alpha_beta['benchmark'].str.contains('50|Nifty 50', case=False, na=False)]
    if not nifty_50_ab.empty:
        avg_alpha = nifty_50_ab['alpha'].mean()
        avg_beta = nifty_50_ab['beta'].mean()
        
        insights += f"""

## Benchmark Analysis (Nifty 50)

5. **Alpha & Beta Metrics**:
   - Average Alpha (annualized): {avg_alpha*100:.2f}% (excess returns vs Nifty 50)
   - Average Beta: {avg_beta:.2f} (systematic risk multiplier)
   - Beta < 1.0 indicates lower market sensitivity; Beta > 1.0 indicates higher sensitivity

6. **Alpha Leaders**:
   - Top alpha generation indicates superior fund management skill
   - Positive alpha after fees is the key indicator of outperformance
"""

insights += """

## Investor Demographics Insights

7. **Market Reach**:
   - Geographic concentration varies; top states likely represent major metros
   - Opportunity to expand in under-penetrated tier 2/3 cities

8. **Investment Pattern**:
   - SIP vs Lumpsum distribution indicates investor sophistication
   - SIP-heavy portfolios suggest long-term wealth building focus

9. **Income Distribution**:
   - Higher income brackets typically have larger AUMs
   - Opportunity in mass affluent segment

## Recommendations

1. **Portfolio Construction**: Use top Sharpe ratio funds as core holdings
2. **Risk Management**: Consider VaR and max drawdown in client suitability assessments
3. **Benchmark Selection**: Choose appropriate benchmarks (Nifty 50/100) based on fund category
4. **Alpha Generation**: Identify funds consistently generating positive alpha
5. **Market Expansion**: Focus on tier 2/3 cities and higher income segments
"""

print(insights)

# Save insights
with open(REPORTS / 'Day_6_Insights.md', 'w') as f:
    f.write(insights)

print('\nInsights saved to Day_6_Insights.md')

## Save Output Files

In [ ]:
# Save all output files
print('Saving output files...')

# 1. Fund Sharpe Ranks
if not fund_sharpe_ranks.empty:
    output_cols = ['amfi_code', 'scheme_name', 'fund_house', 'annualized_return', 
                   'annualized_volatility', 'sharpe_ratio', 'sortino_ratio', 'var_95', 
                   'max_drawdown', 'sharpe_rank', 'sortino_rank', 'var_rank']
    output_cols = [col for col in output_cols if col in fund_sharpe_ranks.columns]
    fund_sharpe_ranks[output_cols].to_csv(REPORTS / 'fund_sharpe_ranks.csv', index=False)
    print('✓ fund_sharpe_ranks.csv saved')

# 2. VaR & Drawdown Summary
if not fund_sharpe_ranks.empty:
    var_dd_cols = ['amfi_code', 'scheme_name', 'fund_house', 'var_95', 'max_drawdown', 
                   'drawdown_start_date', 'drawdown_end_date']
    var_dd_cols = [col for col in var_dd_cols if col in fund_sharpe_ranks.columns]
    fund_sharpe_ranks[var_dd_cols].to_csv(REPORTS / 'var_drawdown_summary.csv', index=False)
    print('✓ var_drawdown_summary.csv saved')

# 3. Alpha & Beta Table
if not alpha_beta.empty:
    alpha_beta_with_info = alpha_beta.copy()
    if not scheme_performance.empty:
        scheme_info = scheme_performance[['amfi_code', 'scheme_name']].drop_duplicates()
        alpha_beta_with_info = alpha_beta_with_info.merge(scheme_info, on='amfi_code', how='left')
    
    output_ab_cols = ['amfi_code', 'scheme_name', 'benchmark', 'alpha', 'beta', 
                      'tracking_error', 'r_squared']
    output_ab_cols = [col for col in output_ab_cols if col in alpha_beta_with_info.columns]
    alpha_beta_with_info[output_ab_cols].to_csv(REPORTS / 'alpha_beta_table.csv', index=False)
    print('✓ alpha_beta_table.csv saved')

# 4. Rolling Correlations
if not rolling_correlations.empty:
    rolling_correlations.to_csv(REPORTS / 'rolling_correlations.csv', index=False)
    print('✓ rolling_correlations.csv saved')

print('\nAll output files saved to:', REPORTS)

In [ ]:
# Summary statistics
print('\n' + '='*60)
print('DAY 6 ADVANCED ANALYTICS - SUMMARY')
print('='*60)

print(f'\nFunds Analyzed: {len(fund_sharpe_ranks)}')
if not fund_sharpe_ranks.empty:
    print(f'Average Sharpe Ratio: {fund_sharpe_ranks["sharpe_ratio"].mean():.2f}')
    print(f'Average Sortino Ratio: {fund_sharpe_ranks["sortino_ratio"].mean():.2f}')
    print(f'Average Annual Return: {fund_sharpe_ranks["annualized_return"].mean()*100:.2f}%')
    print(f'Average Annual Volatility: {fund_sharpe_ranks["annualized_volatility"].mean()*100:.2f}%')
    print(f'Average Max Drawdown: {fund_sharpe_ranks["max_drawdown"].mean()*100:.2f}%')
    print(f'Average VaR 95%: {fund_sharpe_ranks["var_95"].mean()*100:.2f}%')

print(f'\nAlpha-Beta Records: {len(alpha_beta)}')
if not rolling_correlations.empty:
    print(f'Correlation Records: {len(rolling_correlations)}')

if investor_data is not None and len(investor_data) > 0:
    print(f'\nInvestor Records Analyzed: {len(investor_data)}')

print(f'\nOutput Files Generated:')
for file in REPORTS.glob('*.csv'):
    print(f'  - {file.name}')

for file in REPORTS.glob('*.html'):
    print(f'  - {file.name}')

print(f'\nMarkdown Reports:')
for file in REPORTS.glob('*.md'):
    print(f'  - {file.name}')

print('\n' + '='*60)